In [6]:
%matplotlib widget
%load_ext autoreload
%autoreload 2
import os, sys
dir2 = os.path.abspath('')
dir1 = os.path.dirname(dir2)
if not dir1 in sys.path: sys.path.append(dir1)

import warnings
warnings.filterwarnings("ignore", message="findfont: Font family")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
START_DEFLECTION = 0.1
START_VELOCITY = 0  
MASS = 1  
DEFAULT_C = 30
DEFAULT_D = 0.1  
OMEGA = 6.3
ALPHA = 0.5
M_U = 10
EPSILON = 1

In [8]:
import numpy as np
from scipy.integrate import odeint

from src.project.calculations.int_solver import validate_solution

correct = False


class StudentSolver:
    def __init__(self) -> None:
        return None

    def state_space_steady(self, x, t, m_u, m, d, c, e, omega):
        b2 = -m_u / m
        omega_0 = np.sqrt(c / m)
        delta = d / (2 * m)
        # write your code here 
        [z, z_p] = x
        x_p = [
            z_p,
            -2 * delta * z_p - omega_0**2 * z + b2 * e * omega**2 * np.sin(omega * t),
        ]
        return x_p

    def state_space_accelerated(self, x, t, m_u, m, d, c, e, alpha):
        delta = d / (2 * m)
        omega_0 = np.sqrt(c / m)
        b2 = -m_u / m
        [z, z_p] = x
        x_p = [
            z_p,
            -2 * delta * z_p
            - omega_0**2 * z
            - b2 * e * (alpha * t) ** 2 * np.sin(0.5 * alpha * t**2),
        ]
        return x_p

    def integrate(self, func, start_deflection, start_velocity, t, *args):

        x0 = (start_deflection, start_velocity)
        return odeint(func=func, y0=x0, t=t, args=args)[:, 0]


from src.project.calculations.int_solver import IntSolverAufgabe1Uebung3 as IntSolverCorrect

np.set_printoptions(precision=4, suppress=True, linewidth=120)
student_solver = StudentSolver()
correct_solver = IntSolverCorrect()
t = np.linspace(0, 20, 500)


# settled
settled_student = student_solver.integrate(
                                student_solver.state_space_steady, 
                                START_DEFLECTION,
                                START_VELOCITY,
                                t,
                                M_U,
                                MASS,
                                DEFAULT_D, 
                                DEFAULT_C,
                                EPSILON,
                                ALPHA)

settled_correct = correct_solver.integrate(
                                student_solver.state_space_steady, 
                                START_DEFLECTION,
                                START_VELOCITY,
                                t,
                                M_U,
                                MASS,
                                DEFAULT_D,
                                DEFAULT_C,
                                EPSILON,
                                ALPHA)

# accelerated
accelerated_student = student_solver.integrate(
                                student_solver.state_space_accelerated, 
                                START_DEFLECTION,
                                START_VELOCITY,
                                t,
                                M_U, MASS, DEFAULT_D, DEFAULT_C, EPSILON, OMEGA)

accelerated_correct = correct_solver.integrate(
                                correct_solver.state_space_accelerated,
                                START_DEFLECTION,
                                START_VELOCITY,
                                t,
                                M_U, MASS, DEFAULT_D, DEFAULT_C, EPSILON, OMEGA)


# test results
print("Settled: ")
correct_settled = validate_solution(settled_correct, settled_student, relative_threshold=0.05)
print()
print("Accelerated: ")
correct_acc = validate_solution(accelerated_correct, accelerated_student, relative_threshold=0.05)

# solution is correct if both settled and accelerated are correct
correct = correct_settled and correct_acc

Settled: 
Yor solution is correct!

Accelerated: 
Yor solution is correct!


In [9]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

from src.project.gui.gui_a1_u3 import GUI
from src.project.anim.animations.a1_u3_animation import Aufgabe1

if correct: 
    animation_instance = Aufgabe1(calculator=student_solver)
    app = GUI(default_c=DEFAULT_C,
            default_d=DEFAULT_D,
            animation_instance=animation_instance)
    display(app.app_layout)
else: 

    print("Your solution is not correct. Please check your implementation.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


c:\Users\shredder\VSCodeProjects\Technische-Mechanik\venv\Lib\site-packages\scipy\signal\_ltisys.py:2202: RuntimeWarning: divide by zero encountered in log10
  mag = 20.0 * np.log10(abs(y))
c:\Users\shredder\VSCodeProjects\Technische-Mechanik\venv\Lib\site-packages\scipy\signal\_filter_design.py:187: RuntimeWarning: divide by zero encountered in divide
  h = polyval(b, s) / polyval(a, s)
c:\Users\shredder\VSCodeProjects\Technische-Mechanik\venv\Lib\site-packages\scipy\signal\_filter_design.py:187: RuntimeWarning: invalid value encountered in divide
  h = polyval(b, s) / polyval(a, s)


AppLayout(children=(GridspecLayout(children=(HTML(value='<strong style="font-family: Arial, sans-serif;">Varia…

In [10]:
# Setup path
dir2 = os.path.abspath('')
dir1 = os.path.dirname(dir2)
if dir1 not in sys.path:
    sys.path.append(dir1)

    
import os, sys
from ipycanvas import MultiCanvas
from src.project.utils.helper import (
    abs_value,
    draw_line_with_strokes,
    ghetto_feder_daempfer_element_top,
    ghetto_feder_daempfer_element_bottom
)
from src.project.utils.ext_utils.spring import spring_module



# Create multi-layer canvas
canvas_width, canvas_height = 600, 600
n_layers = 5
multi_canvas = MultiCanvas(n_canvases=n_layers, width=canvas_width, height=canvas_height)
bg_layer     = multi_canvas[0]
rect_layer   = multi_canvas[1]
circ_layer   = multi_canvas[2]
spring_layer = multi_canvas[3]
text_layer   = multi_canvas[4]

# === Values ===
rect_x = abs_value(canvas_width, 30)
rect_y = abs_value(canvas_height, 45)
rect_w = abs_value(canvas_width, 25)
rect_h = rect_w

# === Layer 0: Background Frame with Strokes ===
bg_layer.line_width = 1.5

# Frame left
left_x1 = rect_x - abs_value(canvas_width, 2)
left_y1 = rect_y
left_x2 = left_x1
left_y2 = left_y1 + abs_value(canvas_height, 50)

# Frame bottom
bottom_x1 = left_x2
bottom_y1 = left_y2
bottom_x2 = bottom_x1 + rect_w + abs_value(canvas_width, 4)
bottom_y2 = bottom_y1

# Frame right
right_x1 = bottom_x2
right_y1 = bottom_y2
right_x2 = right_x1
right_y2 = left_y1

draw_line_with_strokes(bg_layer, right_x1, right_y1, right_x2, right_y2,
                       len_strokes=abs_value(canvas_width, 5), num_strokes=180,
                       angle=5, direction_strokes="right")

draw_line_with_strokes(bg_layer, bottom_x1, bottom_y1, bottom_x2, bottom_y2,
                       len_strokes=abs_value(canvas_width, 5), num_strokes=10,
                       angle=30, direction_strokes="bottom")

draw_line_with_strokes(bg_layer, left_x1, left_y1, left_x2, left_y2,
                       len_strokes=abs_value(canvas_width, 5), num_strokes=10,
                       angle=30, direction_strokes="left")

# === Layer 1: Rectangle ===
rect_layer.line_width = 1.5
rect_layer.fill_style = "#bebebe"
rect_layer.fill_rect(rect_x, rect_y, rect_w, rect_h)
rect_layer.stroke_rect(rect_x, rect_y, rect_w, rect_h)

# === Layer 2: Circle and Connecting Lines ===
circ_r = abs_value(canvas_height, 13)
circ_x = rect_x + rect_w / 2
circ_y = rect_y - abs_value(canvas_height, 15)

circ_layer.stroke_line(rect_x, rect_y, circ_x, circ_y)
circ_layer.stroke_line(rect_x + rect_w, rect_y, circ_x, circ_y)

circ_layer.fill_style = "white"
circ_layer.line_width = 1.5
circ_layer.stroke_circle(circ_x, circ_y, circ_r)
circ_layer.fill_circle(circ_x, circ_y, circ_r)

circ_layer.line_width = 1.0
circ_layer.stroke_circle(circ_x, circ_y, abs_value(canvas_width, 1))
circ_layer.fill_circle(circ_x, circ_y, abs_value(canvas_width, 1))

# === Layer 3: Spring-Damper System ===
spring_layer.line_width = 1.5

# Top Fork
anker_point_top = (bottom_x1 + (rect_w + abs_value(canvas_width, 4)) / 2, bottom_y1)
spring_anker_point_top = ghetto_feder_daempfer_element_top(
    spring_layer, anker_point_top,
    fork_width=abs_value(canvas_width, 4),
    anker_point_extension=abs_value(canvas_width, 2),
    daempfer_fork_extension=abs_value(canvas_width, 3),
    daempfer_fork_length=abs_value(canvas_width, 5),
    daempfer_fork_width=abs_value(canvas_width, 4),
    direction="vertical",
    top_to_bottom=False
)

# Bottom Fork
anker_point_bottom = (rect_x + rect_w / 2, rect_y + rect_h)
spring_anker_point_bottom = ghetto_feder_daempfer_element_bottom(
    spring_layer, anker_point_bottom,
    bottom_fork_extension=abs_value(canvas_width, 2),
    bottom_fork_width=abs_value(canvas_width, 4),
    daempfer_length=abs_value(canvas_width, 4),
    daempfer_width=abs_value(canvas_width, 3),
    direction="vertical",
    top_to_bottom=False
)

# Spring
spring_nodes = 20
spring_width = abs_value(canvas_width, 2)
x_coords, y_coords = spring_module.spring(
    spring_anker_point_top,
    spring_anker_point_bottom,
    spring_nodes,
    spring_width,
)
spring_module.draw_spring(
    canvas=spring_layer,
    x_coords=x_coords,
    y_coords=y_coords,
    spring_anker_point=spring_anker_point_top,
    width_offset=0,
    height_offset=0,
    clear_x=canvas_width,
    clear_y=canvas_height
)

# === Layer 4: Text ===
text_layer.fill_style = "black"
text_layer.font = f"{abs_value(canvas_height, 5)}px sans-serif"
text_layer.fill_text("m₀",
    rect_x + rect_w / 2 - abs_value(canvas_width, 3),
    rect_y + rect_h / 2 + abs_value(canvas_height, 2)
)
text_layer.fill_text("c",
    rect_x + abs_value(canvas_width, 5),
    rect_y + abs_value(canvas_width, 40)
)
text_layer.fill_text("d",
    rect_x + abs_value(canvas_width, 18),
    rect_y + abs_value(canvas_width, 40)
)

# === Display all layers ===
display(multi_canvas)


MultiCanvas(height=600, width=600)